# Project 1: Recommendation under drift

This notebook is the starting point of the project; the subject (`projects.pdf`, section 1)
gives the context and the grading. You hand in this notebook, completed: it must run from
top to bottom in less than 15 minutes on a laptop.

The project has three guided tasks, then a part where you choose how to go further:

1. **Forgetting is necessary**: run LinUCB on the drifting environment and explain its regret.
2. **A sliding window**: implement LinUCB on the last $w$ observations, and choose $w$.
3. **Choosing offline**: estimate the click rate of the agents from logs, by replay, and
   compare with the online ranking.
4. **Going further**: a direction of your choice (see the end of the notebook).

Each task ends with a figure or a table and a short written answer (a few sentences) in the
cell "Answer". Everything outside the cells marked `...  # TODO` is given and working.

This notebook needs the version of `seqlearn` that contains `projects/`: if the imports below
fail, download the setup archive again from the course page and replace the folder
`seqlearn/`.

In [ ]:
from collections import deque

import matplotlib.pyplot as plt
import numpy as np

from seqlearn import UCBAgent
from seqlearn.linear import ContextFree, LinearAgent
from seqlearn.plotting import COLORS, MUTED, plot_mean, set_style
from seqlearn.projects.drift import DIM, HORIZON, N_ITEMS, make_env, uniform_log_stream

set_style()

## 0. The simulator

At each round, a user arrives with a profile $u_t$ over $5$ moods, and the agent recommends
one of $10$ items. Item $a$ is clicked with probability $\langle u_t, w_a \rangle$, where
$w_a$ is the appeal of the item, as in the track `users` of Lab 4. In the notation of the
linear bandits (session 3), the feature vector of item $a$ is the profile written in the
block $a$ of a vector of dimension $d = 50$, and the unknown parameter $\theta$ stacks the
$10$ appeals.

The appeals change at $4$ unknown rounds (`env.change_points`, for the plots only), where
about half of the items get new appeals. The horizon is $T = 20\,000$.

In [ ]:
env = make_env(rng=0)
x = env.features()
print("features:", x.shape, " change points:", env.change_points)
print("profile of the first user:", np.round(env.profile, 2))
print("click probabilities of the 10 items:", np.round(env.means(), 2))

## Given: LinUCB, the runs and the plots

`LinUCB` is the agent of Lab 3, written with the matrix $V = \lambda I + \sum_s x_s x_s^\top$
and the vector $b = \sum_s r_s x_s$. `simulate` plays an agent on several seeds and returns
the cumulative regret and the click rate of each run.

In [ ]:
class LinUCB(LinearAgent):
    def __init__(self, dim, alpha=1.0, lam=1.0, rng=None):
        self.alpha, self.lam = alpha, lam
        super().__init__(dim, rng)

    def reset(self):
        super().reset()
        self.V = self.lam * np.eye(self.dim)
        self.b = np.zeros(self.dim)

    def act(self, features):
        V_inv = np.linalg.inv(self.V)
        theta_hat = V_inv @ self.b
        widths = np.sqrt(np.einsum("ij,jk,ik->i", features, V_inv, features))
        return int(np.argmax(features @ theta_hat + self.alpha * widths))

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        self.V += np.outer(x, x)
        self.b += reward * x


def simulate(make_agent, n_seeds=5, horizon=HORIZON):
    """Play the agent built by make_agent() on the environments of seeds 0 to n_seeds - 1.
    Return the cumulative regrets, of shape (n_seeds, horizon), and the click rates."""
    regrets, rates = np.empty((n_seeds, horizon)), np.empty(n_seeds)
    for seed in range(n_seeds):
        env, agent = make_env(rng=seed), make_agent()
        agent.reset()
        regret, clicks = 0.0, 0.0
        for t in range(horizon):
            features = env.features()
            arm = agent.act(features)
            means = features @ env.theta
            reward = env.pull(arm)
            agent.update(features[arm], arm, reward)
            regret += means.max() - means[arm]
            clicks += reward
            regrets[seed, t] = regret
        rates[seed] = clicks / horizon
    return regrets, rates


def plot_regrets(results, title):
    """results: dictionary name -> (regrets, rates), as returned by simulate. Left: the run
    of seed 0, with its change points (dotted lines). Right: the mean over the seeds, with
    a band of 2 standard errors (the change points differ from one seed to the other)."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for (name, (regrets, rates)), color in zip(results.items(), COLORS):
        axes[0].plot(regrets[0], color=color, label=name)
        plot_mean(axes[1], regrets, label=f"{name} (click rate {rates.mean():.3f})", color=color)
    for c in make_env(rng=0).change_points:
        axes[0].axvline(c, color=MUTED, linestyle=":", linewidth=1)
    axes[0].set(xlabel="round", ylabel="cumulative regret", title=f"{title}: seed 0")
    axes[1].set(xlabel="round", title=f"{title}: mean over {len(regrets)} seeds")
    axes[1].legend(loc="upper left")
    plt.show()

## Task 1: forgetting is necessary

Run LinUCB ($\alpha = 1$) and UCB1 without context (`ContextFree(UCBAgent(N_ITEMS))`) on $5$
seeds and plot their regrets.

**Expected**: the figure, and an answer to: why does the regret of LinUCB grow faster after
a change point, and why does it not recover? Relate it to the estimate
$\hat\theta = V^{-1} b$.

**To read**: slides of session 3 (LinUCB, confidence ellipsoid).

In [ ]:
results = {}
...  # TODO
plot_regrets(results, "Task 1")

**Answer.** *Write your answer here.*

## Task 2: a sliding window

Complete `WindowLinUCB`: the same agent as `LinUCB`, but $V$ and $b$ only contain the last
$w$ observations. Keep the observations in a `deque`; when there are more than $w$ of them,
remove the oldest one from $V$ and $b$.

Then compare LinUCB with windows $w \in \{500, 2000, 5000\}$ on $5$ seeds.

**Expected**: the figure, and an answer to: which window is best, and why are a too small
and a too large window both bad? (Think of the bias and the variance of $\hat\theta$.)

**To read**: Garivier and Moulines, *On upper-confidence bound policies for switching bandit
problems*, ALT 2011 (sliding-window UCB, section 4); Russac, Vernade and Cappé, *Weighted
linear bandits for non-stationary environments*, NeurIPS 2019 (section 1 and 2).

In [ ]:
class WindowLinUCB(LinUCB):
    def __init__(self, dim, window, alpha=1.0, lam=1.0, rng=None):
        self.window = window
        super().__init__(dim, alpha, lam, rng)

    def reset(self):
        super().reset()
        self.memory = deque()

    def update(self, x, arm, reward):
        ...  # TODO

In [ ]:
windows = [500, 2000, 5000]
results2 = {"LinUCB": results["LinUCB"]}
...  # TODO
plot_regrets(results2, "Task 2")

**Answer.** *Write your answer here.*

## Task 3: choosing offline

In practice, $w$ cannot be tuned on real users. The logs of a uniformly random recommender
allow to evaluate an agent offline, by **replay** (Li et al., 2011): go through the logs in
order; when the agent chooses the logged item, it receives the logged click (and is
updated); otherwise, the log is skipped. The estimate is the mean click over the kept logs.

`uniform_log_stream(n, rng)` generates $n$ logs one at a time, as tuples
`(features, item, click)`, on a new environment whose $4$ change points are spread over the
$n$ logs. Complete `replay`, then estimate the click rate of LinUCB and of the three windows
on $2$ streams of $n = 20\,000$ logs, then on one stream of $n = 200\,000$ logs (about $3$
minutes).

**Expected**: the table printed below, and an answer to: with $20\,000$ logs, does the
replay rank the agents as the online runs do? How many logs are kept, and what does it
change for an agent with a window? Why do $200\,000$ logs work better? (Hint: with uniform
logs, the chosen item is the logged one once out of $10$.)

**To read**: slides of session 5 (learning from logs); Li, Chu, Langford and Wang,
*Unbiased offline evaluation of contextual-bandit-based news article recommendation
algorithms*, WSDM 2011 (sections 3 and 4).

In [ ]:
def replay(stream, agent):
    """Replay estimate of the click rate of agent on the logs of stream; also return the
    number of kept logs."""
    agent.reset()
    clicks, kept = 0.0, 0
    for features, item, click in stream:
        ...  # TODO
    return clicks / max(kept, 1), kept

In [ ]:
agents = {"LinUCB": lambda: LinUCB(DIM)} | {f"window {w}": (lambda w=w: WindowLinUCB(DIM, w)) for w in windows}
for n in [20_000, 200_000]:
    print(f"\n{n} logs\n{'agent':<14} {'online':>8} {'replay':>8} {'kept logs':>10}")
    for name, make_agent in agents.items():
        estimates = [replay(uniform_log_stream(n, rng=100 + seed), make_agent()) for seed in range(2 if n < 100_000 else 1)]
        online = results2[name][1].mean()
        print(f"{name:<14} {online:8.3f} {np.mean([e[0] for e in estimates]):8.3f} {np.mean([e[1] for e in estimates]):10.0f}")

**Answer.** *Write your answer here.*

## Going further

Choose one direction, among these suggestions or your own idea: state the question you ask,
then answer it as in the tasks (code, a figure, a short answer). One well-studied direction
is better than several superficial ones.

- A discounted LinUCB (Russac et al., 2019), with weights $\gamma^{t-s}$; compare it with the
  window.
- Linear Thompson sampling (Lab 3) with a window.
- Detect the change points (for instance with a test on the recent clicks) and restart the
  agent.
- The variance of the replay: how many logs are needed for a reliable ranking?